# Stochastic Policy Gradient Methods in the Uncertain Volatility Model
**ArXivist-generated reproduction notebook**  
Paper: [arXiv:2605.06670](https://arxiv.org/abs/2605.06670)  
Generated: 2026-05-15

This notebook walks through the key components of the implementation,
runs a small-scale training loop, and verifies that the setup matches
the paper's reported behavior.

In [ ]:
# Cell 1: Environment check
import sys
import torch
print(f"Python:         {sys.version.split()[0]}")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — mini-training will work; full training will be slow.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

In [ ]:
# Cell 2: Install project in editable mode
import subprocess, os
# Run from notebooks/ directory; project root is one level up
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "..", "--quiet"],
    capture_output=True, text=True, cwd=os.path.join(os.getcwd(), "..")
)
if result.returncode == 0:
    print("Installation successful.")
else:
    print("WARN:", result.stderr[:500])

# Add src to path for imports
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    sys.path.insert(0, repo_root)
print(f"Repo root: {repo_root}")

## Paper Overview

The **Uncertain Volatility Model (UVM)** models a portfolio of $d$ risky assets whose
volatilities (and optionally correlations) are unknown but bounded:
$$\sigma^i \in [\sigma^i_{\min}, \sigma^i_{\max}], \quad \rho^{ij} \in [\rho^{ij}_{\min}, \rho^{ij}_{\max}]$$

The **robust option price** is the worst-case expectation:
$$V(0, x) = \sup_{\alpha \in \mathcal{A}} \mathbb{E}[e^{-rT} g(X^\alpha_T)]$$

**Key innovation:** The authors solve this as a backward actor-critic RL problem using PPO.
At each of $N$ time steps (backward from $T$ to $0$):
- **Actor** learns the worst-case volatility/correlation control $\alpha^*_n(x)$
- **Critic** learns the value function $V_n(x)$

Two policy classes:
1. **Continuous (squashed Gaussian):** for uncertain $\sigma$ AND $\rho$; enforces PSD via C-vine
2. **Bang-bang (Bernoulli):** for $\sigma \in \{\sigma_{\min}, \sigma_{\max}\}^d$; theoretically optimal in $d=1$

## Component 1: C-vine Correlation Parameterization

**Section 3.2.1** — The key challenge in $d \geq 2$ is that the correlation matrix $\rho$ must be
positive semidefinite. Rather than projecting or using Cholesky decompositions numerically, the
authors use the **C-vine parameterization** (Joe 2006; JKL 2009):

Map partial correlations $y \in (-1,1)^{d(d-1)/2}$ to a valid Cholesky factor $L$ such that
$\rho = LL^T$ is PSD with unit diagonal. For $d=3$:
$$\rho_{12} = y_{12}, \quad \rho_{13} = y_{13}, \quad \rho_{23} = y_{23|1}\sqrt{(1-\rho_{12}^2)(1-\rho_{13}^2)} + \rho_{12}\rho_{13}$$

In [ ]:
# Demo: C-vine correlation parameterization
try:
    from spg_uvm.models.vine import CVineCorrelation
    import torch

    d = 3
    B = 4  # batch of 4
    vine = CVineCorrelation(d=d)
    print(vine)

    # Random partial correlations in (-1,1)
    y = torch.rand(B, d*(d-1)//2) * 1.8 - 0.9  # [B, 3]
    L, rho = vine(y)

    print(f"\nPartial correlations y: {y[0].tolist()}")
    print(f"Cholesky L shape:       {L.shape}")
    print(f"Correlation rho shape:  {rho.shape}")
    print(f"\nrho[0] =\n{rho[0].numpy().round(4)}")

    # Verify PSD: all eigenvalues >= 0
    eigs = torch.linalg.eigvalsh(rho)
    print(f"\nMin eigenvalue (should be >= 0): {eigs.min().item():.6f}")
    print(f"Unit diagonal (should be ~1):    {rho.diagonal(dim1=-2,dim2=-1).mean().item():.6f}")
    print("✓ PSD and unit diagonal verified")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Component 2: Actor & Critic Networks

**Section 4.1.3** — Both networks are shallow MLPs with **one hidden layer of 32 ELU units**:
$$x \xrightarrow{\text{LayerNorm}} \xrightarrow{\text{Linear}(d, 32)} \xrightarrow{\text{ELU}} \xrightarrow{\text{Linear}(32, \text{out})} \text{output}$$

- **Actor** output: $m_\theta(x) \in \mathbb{R}^{d(d+1)/2}$ (mean of latent Gaussian)
- **Critic** output: $V_\phi(x) \in \mathbb{R}$ (scalar value estimate)

In [ ]:
# Demo: Actor and Critic networks
try:
    from spg_uvm.models.networks import ActorNetwork, CriticNetwork

    d = 2
    B = 8

    actor = ActorNetwork(d=d, hidden_units=32, policy_type="continuous")
    critic = CriticNetwork(d=d, hidden_units=32)

    print(actor)
    print(critic)

    x = torch.rand(B, d) * 100 + 50  # realistic asset prices ~[50, 150]

    m_theta = actor(x)
    v_phi   = critic(x)

    print(f"\nInput x:               {x.shape}  ({x.min():.1f} – {x.max():.1f})")
    print(f"Actor output m_theta:  {m_theta.shape}  (latent mean in R^{d*(d+1)//2})")
    print(f"Critic output V_phi:   {v_phi.shape}")

    n_params_actor  = sum(p.numel() for p in actor.parameters())
    n_params_critic = sum(p.numel() for p in critic.parameters())
    print(f"\nActor params:  {n_params_actor:,}")
    print(f"Critic params: {n_params_critic:,}")
    print(f"Total per step:{n_params_actor + n_params_critic:,}")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Component 3: Continuous Policy (TUVM Map)

**Section 3.2.1** — The TUVM map squashes the latent Gaussian sample $z$ to the admissible set:
$$\sigma^i = \frac{\sigma^i_{\max}+\sigma^i_{\min}}{2} + \frac{\sigma^i_{\max}-\sigma^i_{\min}}{2}\tanh(z^i_\sigma)$$
$$y_{\rho} = \tanh(z_{\rho}) \xrightarrow{\text{C-vine}} (L, \rho) \text{ (PSD)}$$

The **likelihood ratio** for PPO (Eq. 21) simplifies to a Gaussian ratio in latent space
(Jacobians cancel).

In [ ]:
# Demo: Continuous policy sampling and TUVM map
try:
    from spg_uvm.models.policy import ContinuousPolicy

    d = 2
    sigma_min = [0.1, 0.1]
    sigma_max = [0.2, 0.2]
    policy = ContinuousPolicy(d=d, sigma_min=sigma_min, sigma_max=sigma_max)
    print(policy)

    B = 6
    x = torch.full((B, d), 100.0)  # all assets at 100
    temperature = 0.5  # lambda

    sigma, L, rho, z, m_theta = policy.sample(x, actor, temperature)

    print(f"\nSampled volatility sigma: {sigma.shape}")
    print(f"  Range: [{sigma.min():.4f}, {sigma.max():.4f}]  (bounds: [{sigma_min[0]}, {sigma_max[0]}])")
    print(f"Cholesky L: {L.shape}")
    print(f"Correlation rho: {rho.shape}")
    print(f"\nExample sigma[0]: {sigma[0].tolist()}")
    print(f"Example rho[0]:   {rho[0].tolist()}")
    print("\n✓ All sigma in bounds:", 
          bool((sigma >= torch.tensor(sigma_min)).all() and (sigma <= torch.tensor(sigma_max)).all()))
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Component 4: Log-Euler Scheme

**Eq. (8), Section 2.3** — One time step of the multi-asset GBM:
$$F(x, a, \xi) = x \odot \exp\!\left[\left(r - \tfrac{1}{2}\mathrm{diag}(aa^T)\right)\tfrac{T}{N} + a\sqrt{\tfrac{T}{N}}\,\xi\right]$$
where $a = \mathrm{diag}(\sigma)\cdot L$ and $\xi \sim \mathcal{N}(0, I_d)$.

In [ ]:
# Demo: Log-Euler scheme (one step)
try:
    from spg_uvm.models.dynamics import LogEulerScheme

    d, T, N, r = 2, 1.0, 128, 0.0
    scheme = LogEulerScheme(d=d, T=T, N=N, r=r)
    print(scheme)

    B = 5
    x0 = torch.full((B, d), 100.0)

    # Build action matrix from policy output
    a_mat = scheme.build_action_matrix(sigma, L[:B])  # [B, d, d]
    xi    = torch.randn(B, d)
    x1    = scheme.step(x0, a_mat, xi)

    print(f"\nInput  x0: {x0.shape}  mean={x0.mean():.2f}")
    print(f"Output x1: {x1.shape}  mean={x1.mean():.2f}")
    print(f"dt = T/N = {scheme.dt:.4f}")
    print(f"\nSample path (asset 0): {x0[0,0]:.2f} -> {x1[0,0]:.4f}")
    print(f"Sample path (asset 1): {x0[0,1]:.2f} -> {x1[0,1]:.4f}")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Component 5: Option Payoffs

The paper tests 5 option types. The key one for Section 4.2.1 is the **geometric outperformer**:
$$g(x) = \max\!\left(0,\; \left(\prod_{i=2}^d x^i\right)^{1/(d-1)} - x^1\right)$$
Asset 1 is the benchmark; assets 2..d form the geometric mean basket.

In [ ]:
# Demo: All 5 option payoffs
try:
    from spg_uvm.payoffs import GeoOutperformer, OutperformerSpread, BestOfButterfly, GeoCallSpread, CallSharpe

    d = 2
    x_sample = torch.tensor([[100.0, 110.0], [100.0, 90.0], [100.0, 100.0]])

    payoffs = {
        "GeoOutperformer":    GeoOutperformer(d=2),
        "OutperformerSpread": OutperformerSpread(d=2),
        "GeoCallSpread":      GeoCallSpread(d=2, K1=90, K2=110),
        "BestOfButterfly":    BestOfButterfly(d=2, K1=85, K2=115),
    }

    print(f"Asset prices: {x_sample.tolist()}\n")
    for name, payoff_fn in payoffs.items():
        vals = payoff_fn(x_sample)
        print(f"{name:<20}: {vals.tolist()}")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Component 6: PPO Loss & Correlation Penalty

**Eq. (20)** — PPO clipped surrogate:
$$\mathcal{L}^{\text{PPO}} = -\mathbb{E}\left[\min(r_\theta \cdot \text{Adv},\; \text{clip}(r_\theta, 1-\varepsilon, 1+\varepsilon)\cdot \text{Adv})\right]$$

**Section 4.1.1** — Correlation Huber penalty (applied at deterministic mean, $d \geq 3$):
$$\Psi(\rho) = \frac{\beta}{d(d-1)} \sum_{i<j} \left[\text{Hub}_{\delta}\!\left(\frac{(\rho^{ij}-\rho^{ij}_{\max})_+}{\rho^{ij}_{\max}-\rho^{ij}_{\min}}\right) + \text{Hub}_{\delta}\!\left(\frac{(\rho^{ij}_{\min}-\rho^{ij})_+}{\rho^{ij}_{\max}-\rho^{ij}_{\min}}\right)\right]$$

In [ ]:
# Demo: PPO loss and correlation penalty
try:
    from spg_uvm.training.losses import PPOLoss, CriticLoss, CorrelationPenalty

    B = 16
    ppo = PPOLoss(epsilon=0.2)
    critic_loss_fn = CriticLoss()
    corr_penalty = CorrelationPenalty(rho_min=-0.5, rho_max=0.5, beta=10, delta=0.05)

    ratio     = torch.ones(B) + torch.randn(B) * 0.3
    advantage = torch.randn(B)
    v_pred    = torch.randn(B)
    v_target  = torch.randn(B)

    ppo_val    = ppo(ratio, advantage)
    critic_val = critic_loss_fn(v_pred, v_target)

    # Correlation penalty for d=3
    d = 3
    vine3 = CVineCorrelation(d=3)
    y3 = torch.rand(B, 3) * 1.8 - 0.9
    _, rho3 = vine3(y3)
    pen = corr_penalty(rho3)

    print(f"PPO loss:           {ppo_val.item():.4f}")
    print(f"Critic MSE loss:    {critic_val.item():.4f}")
    print(f"Corr penalty (d=3): {pen.item():.6f}")
    print("\nAll losses are scalars — ✓")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Mini-Training Demo

A reduced-scale run of the backward actor-critic loop (Section 4 Algorithm 1).
Uses N=4 time steps and E=5 epochs — completes in seconds on CPU.

This does **not** reproduce the paper's results (requires N=128, E=500, M=32768 on GPU).
It confirms the training loop runs without errors.

In [ ]:
# Mini-training: build reduced config inline
try:
    from spg_uvm.utils.config import UVMConfig, set_seed

    # Load default config then shrink it for demo
    import os
    config_path = os.path.join(repo_root, "configs", "default.yaml")
    cfg = UVMConfig.from_yaml(config_path)

    # Reduce for demo
    cfg.uvm_params.N = 4
    cfg.training.M = 256
    cfg.training.minibatch_size = 64
    cfg.training.E_first = 5
    cfg.training.E_subsequent = 2
    cfg.evaluation.n_paths_actor_price = 512
    cfg.hardware.device = "cuda" if torch.cuda.is_available() else "cpu"

    set_seed(42)
    print(f"Mini-training config: d={cfg.model.d}, N={cfg.uvm_params.N}, "
          f"M={cfg.training.M}, E_first={cfg.training.E_first}")
    print(f"Device: {cfg.hardware.device}")
    print()
except Exception as e:
    print(f"ERROR building config: {e}")
    import traceback; traceback.print_exc()

In [ ]:
# Run mini-training
try:
    from spg_uvm.training.trainer import SPGUVMTrainer

    device = torch.device(cfg.hardware.device)
    trainer = SPGUVMTrainer(cfg, device)
    results = trainer.train()

    print("\n=== Mini-Training Results ===")
    print(f"Actor price:  {results['actor_price']:.4f}  "
          f"(CI: [{results['actor_ci_lo']:.4f}, {results['actor_ci_hi']:.4f}])")
    print(f"Total time:   {results['total_time_sec']:.1f}s")
    print("\n✓ Training loop completed without errors")
    print("Note: With N=4 and E=5, the price estimate is not meaningful.")
    print("Use full config (N=128, E_first=500) for paper-level results.")
except Exception as e:
    print(f"ERROR during training: {e}")
    import traceback; traceback.print_exc()

## Sigmoid Annealing Schedule

**Section 4.1.2** — Temperature $\lambda$ and entropy coefficient $\gamma$ are
annealed using a sigmoid schedule (Figure 1 of paper). Visualize the schedule here.

In [ ]:
# Visualize annealing schedule
try:
    import matplotlib.pyplot as plt
    from spg_uvm.training.annealing import SigmoidAnnealer

    E = 500  # epochs for first time step

    temp_annealer = SigmoidAnnealer(v_initial=1.0, v_final=0.01, steepness=0.15)
    lr_annealer   = SigmoidAnnealer(v_initial=5e-3, v_final=1e-4, steepness=0.15)

    epochs = list(range(E))
    temps  = [temp_annealer.get_value(e, E) for e in epochs]
    lrs    = [lr_annealer.get_value(e, E) for e in epochs]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, temps)
    ax1.set_title("Temperature λ (continuous policy)")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("λ")
    ax1.set_ylim(0, 1.05); ax1.grid(True, alpha=0.3)
    ax1.axhline(0.01, color='r', linestyle='--', alpha=0.5, label='λ_final=0.01')
    ax1.legend()

    ax2.semilogy(epochs, lrs)
    ax2.set_title("Learning rate (sigmoid decay)")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("LR (log scale)")
    ax2.grid(True, alpha=0.3)

    plt.suptitle("Sigmoid annealing schedules (Section 4.1.2, Figure 1)", fontsize=13)
    plt.tight_layout()
    plt.savefig("annealing_schedule.png", dpi=100, bbox_inches='tight')
    plt.show()
    print("ASSUMED: sigmoid_steepness=0.15 (exact formula not given in paper; see Figure 1)")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()

## Paper Results Reference

Key results from Table 1 (Section 4.2.1) and Table 3 (Section 4.2.3).

In [ ]:
# Paper's reported results (from SIR evaluation_protocol)
paper_results = {
    "Geo-outperformer (d=2, Table 1)": {
        "SPG-UVM actor price": "13.75 ± 0.06",
        "GTU (Goudenège 2024)": "13.75",
        "NNU (Goudenège 2024)": "13.75",
        "params": "σ∈[0.1,0.2], ρ∈[-0.5,0.5], T=1, r=0",
    },
    "Geo-outperformer (d=5, Table 1)": {
        "SPG-UVM actor price": "12.50 ± 0.05",
        "Reference": "12.64",
        "Relative error": "~1.1%",
    },
    "Geo call spread (d=80, Table 3)": {
        "SPG-UVM actor price": "9.51 ± 0.00",
        "Reference (analytic)": "9.51",
        "params": "fixed ρ=0, σ∈[0.1,0.2]",
    },
}

print("=" * 60)
print("PAPER REPORTED RESULTS (arXiv:2605.06670)")
print("=" * 60)
for experiment, vals in paper_results.items():
    print(f"\n{experiment}:")
    for k, v in vals.items():
        print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("To reproduce: python train.py --config configs/default.yaml")
print("Expected runtime: ~30-120 min on NVIDIA RTX 4090 (Section 4.1.3)")
print("=" * 60)

## What To Do Next

### Full Reproduction
```bash
# Table 1: 2D geo-outperformer (uncertain correlations)
python train.py --config configs/default.yaml --policy continuous --d 2

# Table 2: Best-of butterfly (bang-bang policy)
python train.py --config configs/best_of_butterfly.yaml --policy bangbang

# Table 3: High-dimensional geo call spread (d=10, fixed rho)
python train.py --config configs/default.yaml --payoff geo_call_spread --d 10
```

### Implementation Assumptions (from SIR)

| Assumption | Confidence | Note |
|---|---|---|
| Sigmoid annealing formula | 0.72 | Parameterized from Figure 1; tune `sigmoid_steepness` in config |
| Adam β₁=0.9, β₂=0.999 | 0.85 | PyTorch defaults; paper says "Adam" without specifying betas |
| C-vine for d≥4 (recursive) | 0.88 | Verify unit tests for PSD + unit diagonal |

### Stage 6: Compare Your Results
After running full training, feed your `results.json` back to ArXivist's
Results Comparator (Stage 6) to get a reproducibility score and hallucination report.